# Safety-mix reward model — target RewardBench's Safety gap

RewardBench showed the helpfulness-only RMs are *bad* at Safety (0.5B 0.368, 1.5B **0.286** —
`refusals-dangerous` 0.07: they confidently prefer harmful-but-helpful over refusals). The measurable
fix: add **safety preference pairs** to the RM's training data. This trains a **Qwen2.5-0.5B-Instruct**
RM on **4000 cleaned UltraFeedback + 2000 PKU-SafeRLHF** (chosen = the safer response), same 6000-pair
budget, then scores it on **RewardBench**. Win = Safety rises vs the uf-only 0.5B (0.368 / overall 0.550)
without wrecking Chat/Reasoning. ~1.5-2 h train + ~10 min eval on a T4.

In [ ]:
import torch, json
cap=torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
P100=tuple(cap)==(6,0); json.dump({'p100':P100},open('/tmp/gpu.json','w'))
print(f'GPU: {name} cap={cap} -> '+('P100 fp32' if P100 else 'T4 bf16'))

In [ ]:
!pip install -q 'transformers>=4.44,<5' 'datasets>=2.20' 'accelerate>=0.33' 'peft>=0.12' pytest
import json, subprocess, sys
if json.load(open('/tmp/gpu.json'))['p100']:
    subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44,<4.48'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','torch==2.4.1','torchvision==0.19.1',
                    'torchaudio==2.4.1','--index-url','https://download.pytorch.org/whl/cu121'],check=True)

In [ ]:
import os
!rm -rf /kaggle/working/rlhf-pipeline && git clone -q https://github.com/TheYellowDuck/RLHF-pipeline.git /kaggle/working/rlhf-pipeline
%cd /kaggle/working/rlhf-pipeline

In [ ]:
!python -m pytest tests/ -q 2>&1 | tail -2   # includes PKU + RewardBench tests

## Config

In [ ]:
import json
P100=json.load(open('/tmp/gpu.json'))['p100']
MODEL='Qwen/Qwen2.5-0.5B-Instruct'
UF='argilla/ultrafeedback-binarized-preferences-cleaned'
PKU='PKU-Alignment/PKU-SafeRLHF'
MIX=f'{UF}:train[2000:6000],{PKU}:train[:2000]'   # 4000 helpfulness + 2000 safety = 6000
DTYPE,BF16=('float32','false') if P100 else ('bfloat16','true')
print('safety-mix:', MIX)

## Train the safety-mix RM (4000 UF + 2000 PKU-SafeRLHF)

In [ ]:
!python scripts/train_reward_model.py \
  -o model.name_or_path=$MODEL -o model.dtype=$DTYPE \
  -o data.name="$MIX" -o data.train_split='train' -o data.eval_split=none \
  -o data.max_length=512 -o train.epochs=1 -o train.batch_size=4 -o train.grad_accum=4 \
  -o train.bf16=$BF16 -o train.lr=1.0e-5 -o train.label_smoothing=0.0 -o data.max_pair_similarity=1.0 \
  -o output_dir=checkpoints/rm_safety

## Score it on RewardBench (vs uf-only 0.5B: overall 0.550, Safety 0.368)

In [ ]:
import subprocess
cmd='python scripts/eval_rewardbench.py --reward-model checkpoints/rm_safety --device cuda --batch-size 16 --max-length 1024'
print('$',cmd,flush=True)
o=subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(o.stdout[-1500:])
if o.returncode: print('STDERR:', o.stderr[-2000:])
allout=o.stdout+'\n'+o.stderr
rep=[l for l in allout.splitlines() if 'rewardbench report' in l]
md=('# Safety-mix RM (0.5B, 4000 UF + 2000 PKU-SafeRLHF) on RewardBench\n\n'
    'Baseline uf-only 0.5B: overall 0.550 | Chat 0.789 | Chat-Hard 0.413 | Safety 0.368 | Reasoning 0.630\n\n'
    '## This RM\n```\n'+o.stdout[-1200:]+'\n```\n'
    +('\n## Full per-subset\n```\n'+rep[-1][-1600:]+'\n```\n' if rep else '')
    +('\n## stderr\n```\n'+o.stderr[-1500:]+'\n```\n' if o.returncode else ''))
open('/kaggle/working/RESULTS.md','w').write(md); print('\nsaved -> RESULTS.md')

### Read
If **Safety** jumps from 0.368 toward the refusal-aware range while Chat/Reasoning hold, adding
safety preference data is the working lever — and RewardBench proves it without a PPO+judge chain.
Watch the tradeoff: too much safety data can dent Chat/Reasoning (the helpfulness signal).